# Tarea 3 — PyTorch

**Nombre del integrante:** Samantha Rojas

**Cédula del integrante:** 

> Enunciado, rúbrica y nombres de los archivos `.pth` a entregar: ver `README.md` de esta carpeta.
> Antes del commit: *Restart & Run All* debe terminar sin errores y quedar en el menú.

## 1. Imports y constantes

In [ ]:
import copy
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "./data"
NOTEBOOK_NUMBER = 3
BATCH_SIZE = 128
VALIDATION_SIZE = 6000
SEED = 0

INPUT_SIZE = 784
ORIGINAL_CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
SIMPLIFIED_CLASSES = ["Top", "Bottom", "Footwear", "Bag"]

# Fashion MNIST label -> simplified label (0 = Top, 1 = Bottom, 2 = Footwear, 3 = Bag)
SIMPLIFIED_MAP = torch.tensor([0, 1, 0, 1, 0, 2, 0, 2, 3, 2])

CLASS_NAMES = {"original": ORIGINAL_CLASSES, "simplified": SIMPLIFIED_CLASSES}

torch.manual_seed(SEED)
print(f"Dispositivo: {DEVICE}")

## 2. Datos (Fashion MNIST original y clases simplificadas)

In [ ]:
def load_fashion_mnist():
    """Download Fashion MNIST and return (train_x, train_y, test_x, test_y) as tensors."""
    train_set = torchvision.datasets.FashionMNIST(DATA_DIR, train=True, download=True)
    test_set = torchvision.datasets.FashionMNIST(DATA_DIR, train=False, download=True)
    # Dividing by 255 gives the same [0, 1] values as transforms.ToTensor()
    train_x = train_set.data.float() / 255
    test_x = test_set.data.float() / 255
    return train_x, train_set.targets, test_x, test_set.targets


def remap_labels(labels, classification):
    """Convert the original labels to the simplified ones when needed."""
    return SIMPLIFIED_MAP[labels] if classification == "simplified" else labels


def make_batches(x, y, shuffle):
    """Yield (images, labels) mini-batches."""
    order = torch.randperm(len(x)) if shuffle else torch.arange(len(x))
    for start in range(0, len(x), BATCH_SIZE):
        idx = order[start:start + BATCH_SIZE]
        yield x[idx].to(DEVICE), y[idx].to(DEVICE)


train_x, train_y, test_x, test_y = load_fashion_mnist()

# Hold out a validation split of the training set (only used for early stopping)
perm = torch.randperm(len(train_x), generator=torch.Generator().manual_seed(SEED))
val_idx, fit_idx = perm[:VALIDATION_SIZE], perm[VALIDATION_SIZE:]
val_x, val_y = train_x[val_idx], train_y[val_idx]
fit_x, fit_y = train_x[fit_idx], train_y[fit_idx]

print(f"Entrenamiento: {len(fit_x)}, validación: {len(val_x)}, prueba: {len(test_x)}")

## 3. Modelos (rectangular y convolucional)

In [ ]:
def funnel_sizes(hidden_layers, num_outputs):
    """Hidden layer sizes by linear interpolation between the input and output layers."""
    return [round(INPUT_SIZE + (num_outputs - INPUT_SIZE) * i / (hidden_layers + 1))
            for i in range(1, hidden_layers + 1)]


def build_model(hp):
    """Build the nn.Sequential described by the hyperparameters dict."""
    num_outputs = len(CLASS_NAMES[hp["classification"]])
    if hp["network_type"] == "convolutional":
        sizes = funnel_sizes(hp["hidden_layers"], num_outputs)
    elif isinstance(hp["neurons_per_layer"], list):
        sizes = hp["neurons_per_layer"]  # already expanded (loaded from a file)
    else:
        sizes = [hp["neurons_per_layer"]] * hp["hidden_layers"]
    hp["neurons_per_layer"] = sizes

    layers = [nn.Flatten()]
    previous = INPUT_SIZE
    for size in sizes:
        layers.append(nn.Linear(previous, size))
        if hp["batch_norm"]:
            layers.append(nn.BatchNorm1d(size))
        layers.append(nn.ReLU())
        if hp["dropout"] > 0:
            layers.append(nn.Dropout(hp["dropout"]))
        previous = size
    layers.append(nn.Linear(previous, num_outputs))
    return nn.Sequential(*layers).to(DEVICE)


def default_hyperparameters(classification, network_type="rectangular", hidden_layers=2, neurons=512):
    return {
        "network_type": network_type,
        "hidden_layers": hidden_layers,
        "neurons_per_layer": neurons,
        "epochs": 0,
        "classification": classification,
        # Optimizations (all optional, see the menu)
        "optimizer": "adam",
        "learning_rate": 0.001,
        "weight_decay": 0.0,
        "dropout": 0.0,
        "batch_norm": False,
        "early_stopping": False,
        "patience": 3,
    }

## 4. Entrenar, probar, guardar y cargar

In [ ]:
def evaluate_loss(model, x, y, classification, criterion):
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for images, labels in make_batches(x, y, shuffle=False):
            labels = remap_labels(labels.cpu(), classification).to(DEVICE)
            total += criterion(model(images), labels).item() * len(labels)
            count += len(labels)
    return total / count


def train(model, hp, epochs):
    """Train for up to `epochs` epochs and plot the loss of each epoch."""
    classification = hp["classification"]
    criterion = nn.CrossEntropyLoss()
    if hp["optimizer"] == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=hp["learning_rate"],
                                    momentum=0.9, weight_decay=hp["weight_decay"])
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=hp["learning_rate"],
                                     weight_decay=hp["weight_decay"])

    losses = []
    best_val, best_state, bad_epochs = float("inf"), None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        total, count = 0.0, 0
        for images, labels in make_batches(fit_x, fit_y, shuffle=True):
            labels = remap_labels(labels.cpu(), classification).to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(labels)
            count += len(labels)
        losses.append(total / count)
        hp["epochs"] += 1
        print(f"Época {epoch}/{epochs} - loss: {losses[-1]:.4f}")

        if hp["early_stopping"]:
            val_loss = evaluate_loss(model, val_x, val_y, classification, criterion)
            if val_loss < best_val:
                best_val, best_state, bad_epochs = val_loss, copy.deepcopy(model.state_dict()), 0
            else:
                bad_epochs += 1
                if bad_epochs >= hp["patience"]:
                    print(f"Early stopping: la pérdida de validación no mejora hace {hp['patience']} épocas.")
                    break
    if best_state is not None:
        model.load_state_dict(best_state)

    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(losses) + 1), losses, marker="o")
    plt.title("Pérdida por época")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.show()
    return losses


def test(model, hp):
    """Evaluate on the test set: confusion matrix, precision, accuracy and recall."""
    classification = hp["classification"]
    names = CLASS_NAMES[classification]
    n = len(names)
    model.eval()
    predictions = []
    with torch.no_grad():
        for images, _ in make_batches(test_x, test_y, shuffle=False):
            predictions.append(model(images).argmax(dim=1).cpu())
    predicted = torch.cat(predictions).numpy()
    actual = remap_labels(test_y, classification).numpy()

    confusion = np.zeros((n, n), dtype=int)  # rows: actual, columns: predicted
    for a, p in zip(actual, predicted):
        confusion[a, p] += 1

    true_positive = np.diag(confusion)
    precision = true_positive / np.maximum(confusion.sum(axis=0), 1)
    recall = true_positive / np.maximum(confusion.sum(axis=1), 1)
    accuracy = true_positive.sum() / confusion.sum()

    plt.figure(figsize=(7, 6))
    plt.imshow(confusion, cmap="Blues")
    plt.colorbar()
    plt.xticks(range(n), names, rotation=45, ha="right")
    plt.yticks(range(n), names)
    for i in range(n):
        for j in range(n):
            plt.text(j, i, confusion[i, j], ha="center", va="center",
                     color="white" if confusion[i, j] > confusion.max() / 2 else "black")
    plt.title("Matriz de confusión")
    plt.xlabel("Predicho")
    plt.ylabel("Real")
    plt.tight_layout()
    plt.show()

    print(f"{'Clase':<14}{'Precision':>10}{'Recall':>10}")
    for i, name in enumerate(names):
        print(f"{name:<14}{precision[i]:>10.4f}{recall[i]:>10.4f}")
    print(f"{'Promedio':<14}{precision.mean():>10.4f}{recall.mean():>10.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    return accuracy


def save_network(model, hp, path):
    torch.save({"state_dict": model.state_dict(), "hyperparameters": hp}, path)
    print(f"Red guardada en {path}")


def load_network(path):
    checkpoint = torch.load(path, map_location=DEVICE)
    hp = checkpoint["hyperparameters"]
    model = build_model(hp)
    model.load_state_dict(checkpoint["state_dict"])
    print(f"Red cargada de {path}")
    return model, hp

## 5. Menú

La última celda debe dejar el menú corriendo (crear/cargar, entrenar, probar, guardar, salir).

In [ ]:
def ask(prompt, default, cast=str):
    """Ask for a value; an empty answer or an invalid one gives the default."""
    answer = input(f"{prompt} [{default}]: ").strip()
    if not answer:
        return default
    try:
        return cast(answer)
    except ValueError:
        print(f"Valor inválido, se usa {default}.")
        return default


def ask_yes_no(prompt, default):
    answer = input(f"{prompt} (s/n) [{'s' if default else 'n'}]: ").strip().lower()
    return default if not answer else answer.startswith("s")


def create_network():
    classification = "simplified" if ask("Clasificación (original/simplified)", "original") == "simplified" else "original"
    layers = ask("Número de capas ocultas", 2, int)
    network_type = "convolutional" if ask("Tipo (rectangular/convolutional)", "rectangular") == "convolutional" else "rectangular"
    neurons = 512
    if network_type == "rectangular":
        neurons = ask("Neuronas por capa", 512, int)
    hp = default_hyperparameters(classification, network_type, layers, neurons)
    hp["batch_norm"] = ask_yes_no("¿Usar batch normalization?", False)
    hp["dropout"] = ask("Probabilidad de dropout (0 = sin dropout)", 0.0, float)
    return build_model(hp), hp


def optimization_menu(hp):
    """Optimizations that do not change the architecture."""
    hp["optimizer"] = "sgd" if ask("Optimizador (adam/sgd)", hp["optimizer"]) == "sgd" else "adam"
    hp["learning_rate"] = ask("Learning rate", hp["learning_rate"], float)
    hp["weight_decay"] = ask("Weight decay", hp["weight_decay"], float)
    hp["early_stopping"] = ask_yes_no("¿Usar early stopping?", hp["early_stopping"])
    if hp["early_stopping"]:
        hp["patience"] = ask("Paciencia (épocas)", hp["patience"], int)


def network_menu(model, hp):
    while True:
        print("\n--- Red actual:", hp["network_type"], hp["classification"],
              "| capas ocultas:", hp["hidden_layers"], "| neuronas:", hp["neurons_per_layer"],
              "| épocas:", hp["epochs"], "---")
        print("1. Entrenar")
        print("2. Probar")
        print("3. Guardar")
        print("4. Optimizaciones")
        print("5. Volver al menú principal")
        option = input("Opción: ").strip()
        if option == "1":
            train(model, hp, ask("Número de épocas", 10, int))
        elif option == "2":
            test(model, hp)
        elif option == "3":
            default_path = f"pytorch_fashion_{NOTEBOOK_NUMBER}_{hp['classification']}_{hp['network_type']}.pth"
            save_network(model, hp, ask("Archivo", default_path))
        elif option == "4":
            optimization_menu(hp)
        elif option == "5":
            return
        else:
            print("Opción inválida.")


def main_menu():
    while True:
        print("\n=== Fashion MNIST con PyTorch ===")
        print("1. Crear una nueva red")
        print("2. Cargar una red de un archivo")
        print("3. Salir")
        option = input("Opción: ").strip()
        if option == "1":
            network_menu(*create_network())
        elif option == "2":
            path = ask("Archivo .pth", f"pytorch_fashion_{NOTEBOOK_NUMBER}_original_rectangular.pth")
            if os.path.exists(path):
                network_menu(*load_network(path))
            else:
                print("El archivo no existe.")
        elif option == "3":
            return
        else:
            print("Opción inválida.")


main_menu()